# PDFをOCRしてテキストに保存する

PyMuPDF内蔵のTesseract OCRを使います。各ページを300 dpiでOCRし、日本語と英語を読み取ってUTF-8のtxtファイルに保存します。

上から順にセルを実行してください。最初の2つのコードセルは初回準備用です。パッケージのインストールと学習データのダウンロードにはインターネット接続が必要です。

In [2]:
%pip install pymupdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from pathlib import Path
from urllib.request import urlretrieve

# ノートブックの作業ディレクトリにOCR用の学習データを保存する
# WindowsのTesseractに日本語を含む絶対パスを渡さないよう、相対パスを使う
tessdata_dir = Path("tessdata")
tessdata_dir.mkdir(exist_ok=True)

for language in ["jpn", "eng"]:
    url = f"https://raw.githubusercontent.com/tesseract-ocr/tessdata_fast/main/{language}.traineddata"
    urlretrieve(url, tessdata_dir / f"{language}.traineddata")

## 変換関数

`pdf_to_txt(input_pdf_path, output_txt_path)` に入力PDFと出力txtのパスを渡します。`full=True` でページ全体をOCRし、ページ間を空行で区切ります。出力先のフォルダはあらかじめ用意してください。同名のtxtファイルがある場合は上書きします。

In [16]:
import pymupdf


def pdf_to_txt(input_pdf_path, output_txt_path):
    """PDFの全ページをOCRし、認識したテキストをUTF-8で保存する。"""
    page_texts = []

    with pymupdf.open(input_pdf_path) as pdf:
        for page in pdf:
            ocr_textpage = page.get_textpage_ocr(
                language="jpn+eng",
                dpi=300,
                full=True,
                tessdata=str(tessdata_dir),
            )
            text = page.get_text("text", textpage=ocr_textpage)
            page_texts.append(text.strip())

    Path(output_txt_path).write_text("\n\n".join(page_texts), encoding="utf-8")

## 実行例

作業ディレクトリをこのノートブックのある `デザイン/code` にして、上から順に実行してください。次のセルでは指定した大学の2024〜2026年の英語PDFを変換します。出力先の大学別フォルダも作成します。

In [ ]:
univ_name = ["kyotu"]

for university_name in univ_name:
    output_dir = Path("../ocred_txt") / university_name
    output_dir.mkdir(parents=True, exist_ok=True)

    for year in ["2024", "2025", "2026"]:
        input_pdf_path = Path(f"../data2/{university_name}_kakomon/english/{year}.pdf")
        output_txt_path = output_dir / f"{year}.txt"
        pdf_to_txt(input_pdf_path, output_txt_path)
        print(f"保存しました: {output_txt_path}")


保存しました: ..\ocred_txt\kyotu\2024.txt
保存しました: ..\ocred_txt\kyotu\2025.txt
保存しました: ..\ocred_txt\kyotu\2026.txt


In [40]:
import re
from pathlib import Path


def clean_ocr_text(input_txt_path, output_txt_path):
    text = Path(input_txt_path).read_text(encoding="utf-8")

    # ひらがな・カタカナ・漢字・日本語の句読点を削除する
    text = re.sub(r"[ぁ-ゖゝ-ゟァ-ヺヽ-ヿー㐀-鿿々〆〇、。「」『』【】・（）()@＠|\[\]①②③④⑤⑥⑦⑧⑨\-\=ー＝]", "", text)

    cleaned_lines = []

    for line in text.splitlines():
        # 行頭・行末の空白を削除し、行内の連続する空白を1つにする
        line = " ".join(line.split())

        if line:
            cleaned_lines.append(line)

    Path(output_txt_path).write_text(
        "\n".join(cleaned_lines),
        encoding="utf-8",
    )



In [41]:
clean_name=["housei","keio","nitidai"]
for u_name in clean_name:
    for year in [2024,2025,2026]:
        input_fn=f"../ocred_txt/{u_name}/{year}.txt"
        output_fn=f"../ocred_txt/{u_name}/{year}_english.txt"
        clean_ocr_text(input_fn,output_fn)

OCRの認識結果には誤字が含まれることがあります。数式・表・段組みは元の配置を再現できないことがあります。

参考資料

- [PyMuPDFのOCR API](https://pymupdf.readthedocs.io/en/latest/page.html#Page.get_textpage_ocr)
- [Tesseractの学習データ](https://github.com/tesseract-ocr/tessdata_fast)